# Music Annotation Project V8.8C
## Notebook 54: Strict Notebook 53 vs Notebook 51R2 Exact-Equivalence Audit

**Runtime:** CPU. Select **Runtime → Run all**.

This is a read-only, fail-closed equivalence audit. It compares the completed Notebook 53 registered-replay run directly with the completed Notebook 51R2 durable run.

### Non-negotiable contract
- Notebook 51R2 is the golden master.
- Notebook 53 must have passed its registered-replay verdict.
- Notebook 51R2 must be complete and valid.
- Page roster equality is exact.
- M3, SAFE_REMOVE and M5_REGISTERED_REPLAY use zero-tolerance, pixel-for-pixel equality.
- The Notebook 53 corrected page is compared directly with Notebook 51R2 `M5 / R2_telea`.
- No metric tolerance, IoU threshold or “close enough” rule is permitted.
- Stages use first-failure semantics: after the first failed stage, later stages are recorded as `NOT_RUN_AFTER_FIRST_FAILURE`.
- All evidence is written before the optional final `RuntimeError`.
- No clean truth is loaded or required.
- PASS authorises shadow integration work only. It does not authorise production output or replacement of Notebook 09.


In [70]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy opencv-python-headless pyarrow

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration, output tree and helpers

In [71]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, sys, platform, traceback
import numpy as np
import pandas as pd
import cv2

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N51_ROOT=ROOT/'observed_print_classifier_v8_7_notebook51'
N53_ROOT=ROOT/'observed_print_classifier_v8_8_notebook53R3'
OUTROOT=ROOT/'m5_remove_exact_equivalence_v8_8c'

NOTEBOOK51_RUN_OVERRIDE=None
NOTEBOOK53_RUN_OVERRIDE=None
RAISE_ON_FAIL=True

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook54_v8_8c_exact_equivalence'
OUT=OUTROOT/RUN
TABLES=OUT/'tables'; AUDIT=OUT/'audit'; DIFFS=OUT/'difference_images'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,AUDIT,DIFFS,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def read_gray(path):
    im=cv2.imread(str(path),cv2.IMREAD_GRAYSCALE)
    require(im is not None,f'Unreadable image: {path}')
    return im

def exact_row(stage,page,artifact,golden,candidate):
    same_shape=golden.shape==candidate.shape
    if same_shape:
        xor=np.logical_xor(golden.astype(bool),candidate.astype(bool)) if artifact in {'M3','SAFE_REMOVE','M5_REMOVE'} else golden!=candidate
        different=int(xor.sum())
        max_abs=int(np.max(np.abs(golden.astype(np.int16)-candidate.astype(np.int16)))) if golden.size else 0
        exact=different==0
    else:
        different=np.nan; max_abs=np.nan; exact=False
    return {'stage':stage,'page':page,'artifact':artifact,'golden_shape':str(golden.shape),'candidate_shape':str(candidate.shape),'shape_exact':same_shape,'different_pixels':different,'max_abs_difference':max_abs,'golden_semantic_sha256':array_hash(golden),'candidate_semantic_sha256':array_hash(candidate),'exact':bool(exact)}

print('Output:',OUT)


Output: /content/drive/MyDrive/MusicAnnotationProject/m5_remove_exact_equivalence_v8_8c/20261108_172202_notebook54_v8_8c_exact_equivalence


## 2. Resolve the completed runs and freeze audit source identity

In [72]:
REQ51=['01_pre_registration.json','source_record.json','notebook51_verdict.json','experiment_identity.json','04_candidate_lock/candidate_lock.json','04_candidate_lock/candidate_inventory.csv','05_tables/mask_arm_inventory.csv','07_audit/page_completion.csv']
REQ53 = ['stage_f_verdict.json','experiment_identity.json','tables/page_equivalence.csv','tables/mask_inventory.csv','tables/regional_masks.csv','pages',]
N51=Path(NOTEBOOK51_RUN_OVERRIDE) if NOTEBOOK51_RUN_OVERRIDE else latest_complete(N51_ROOT,REQ51)
N53=Path(NOTEBOOK53_RUN_OVERRIDE) if NOTEBOOK53_RUN_OVERRIDE else latest_complete(N53_ROOT,REQ53)
require(N51 is not None,'No completed Notebook 51R2 durable run found.')
require(N53 is not None,'No completed Notebook 53 run found.')

v51=json.loads((N51/'notebook51_verdict.json').read_text())
i51=json.loads((N51/'experiment_identity.json').read_text())
s51=json.loads((N51/'source_record.json').read_text())
l51=json.loads((N51/'04_candidate_lock/candidate_lock.json').read_text())
v53=json.loads((N53/'stage_f_verdict.json').read_text())
i53=json.loads((N53/'experiment_identity.json').read_text())

source_identity={
 'notebook51_run':str(N51),'notebook53_run':str(N53),
 'notebook51_verdict_sha256':sha256_file(N51/'notebook51_verdict.json'),
 'notebook51_identity_sha256':sha256_file(N51/'experiment_identity.json'),
 'notebook51_candidate_lock_sha256':sha256_file(N51/'04_candidate_lock/candidate_lock.json'),
 'notebook53_verdict_sha256':sha256_file(N53/'stage_f_verdict.json'),
 'notebook53_identity_sha256':sha256_file(N53/'experiment_identity.json'),
 'notebook51_candidate_lock_id':l51.get('lock_id'),
 'notebook53_implementation_id':i53.get('implementation_id'),
 'created_utc':datetime.now(timezone.utc).isoformat()
}
atomic_json(OUT/'source_identity.json',source_identity)
print(json.dumps(source_identity,indent=2))


{
  "notebook51_run": "/content/drive/MyDrive/MusicAnnotationProject/observed_print_classifier_v8_7_notebook51/20261106_080000_v8_7_notebook51R2",
  "notebook53_run": "/content/drive/MyDrive/MusicAnnotationProject/observed_print_classifier_v8_8_notebook53R3/20261106_210922_v8_8b_notebook53R3",
  "notebook51_verdict_sha256": "e458d9993fffefbdd0d861cf7db8801dc52dc177666348348d9b8bb7ef709cf0",
  "notebook51_identity_sha256": "108f37e0f1375a5f19f1b071c4be917681896e9455145e0e3cb18329686abe50",
  "notebook51_candidate_lock_sha256": "a578f39627d8d6403970a7d96d9b7c2cddcd15ab2ddd55054b79cf3bb1fde9b4",
  "notebook53_verdict_sha256": "440fc010ca5a5afe3a509a14825124592db93369dc17fbbb3dfd40b315cff1c6",
  "notebook53_identity_sha256": "b8d07e639ba8e5714eab293872970a0ca77f5e10b8f392b8a0fca51ee7729bfa",
  "notebook51_candidate_lock_id": "a6698c38f38aeac0cc33f06aa0ad360a7363accf33cb1c6121f657a6a94b5c0b",
  "notebook53_implementation_id": null,
  "created_utc": "2026-11-08T17:22:02.767498+00:00"
}


## 3. Ordered stage engine with first-failure semantics

In [73]:
stage_order = [
    'A_SOURCE_RUNS_COMPLETE_AND_REGISTERED_REPLAY_ELIGIBLE',
    'B_PAGE_ROSTER_EXACT',
    'C_M3_MASK_EXACT',
    'D_SAFE_REMOVE_DECISION_EXACT',
    'E_REGISTERED_REPLAY_M5_MASK_EXACT',
    'F_CORRECTED_PAGE_EXACT_NOTEBOOK51R2_R2_TELEA',
    'G_REPLAY_IMMUTABILITY_AND_INTERNAL_CONTRACTS',
]
stage_records=[]; comparison_rows=[]; first_failed_stage=None

def record_stage(stage,passed,details=None):
    global first_failed_stage
    status='PASS' if passed else 'FAIL'
    if first_failed_stage is not None: status='NOT_RUN_AFTER_FIRST_FAILURE'
    elif not passed: first_failed_stage=stage
    row={'stage':stage,'status':status,'passed':True if status=='PASS' else (False if status=='FAIL' else None),'details':details or {}}
    stage_records.append(row)
    return status=='PASS'

def can_run(stage): return first_failed_stage is None


## 4. Stage A: completed-run and lineage eligibility

In [74]:
checks = {
    'n51_version':
        v51.get('version') == '8.7-notebook51R2',

    'n51_valid_run':
        v51.get('valid_run') is True,

    'n51_identity_version':
        i51.get('version') == '8.7-notebook51R2',

    'n51_candidate_lock_truth_firewall':
        l51.get('truth_loaded') is False,

    'n53_stage_f_pass':
        v53.get('verdict') ==
        'PASS_STAGE_F_REGRESSION_COMPATIBILITY',

    'n53_stage_f_exact':
        v53.get('stage_F_corrected_page_exact') is True,

    'n53_zero_differences':
        int(v53.get('total_different_pixels', -1)) == 0,

    'n53_mode_registered_replay':
        v53.get('mode') == 'REGISTERED_REPLAY'
        and i53.get('mode') == 'REGISTERED_REPLAY',

    'n53_version':
        i53.get('version') ==
        '53R3-v8.8b-notebook51R2-regression-compatibility',

    'n53_telea_radius3':
        int(i53.get('inpaint_radius', -1)) == 3,

    'n53_safe_protect_not_operationally_authorised':
        v53.get('safe_protect_operationally_authorised') is False
        and i53.get('operational_safe_protect_authorised') is False,

    'n53_registered_replay_uses_code2_exclusion':
        v53.get(
            'registered_replay_uses_decision_code_2_exclusion'
        ) is True,
}
record_stage(stage_order[0],all(checks.values()),checks)


True

## 5. Stage B: exact page roster

In [75]:
inv51=pd.read_csv(N51/'04_candidate_lock/candidate_inventory.csv',dtype={'page':str})
inv53=pd.read_csv(N53/'tables/page_equivalence.csv',dtype={'page':str})
roster51=sorted(inv51.page.astype(str).unique())
roster53=sorted(inv53.page.astype(str).unique())
roster_details={'notebook51_pages':len(roster51),'notebook53_pages':len(roster53),'only_notebook51':sorted(set(roster51)-set(roster53)),'only_notebook53':sorted(set(roster53)-set(roster51)),'exact_ordered_roster':roster51==roster53}
if can_run(stage_order[1]): record_stage(stage_order[1],roster51==roster53,roster_details)
else: record_stage(stage_order[1],False,{'not_run_after':first_failed_stage})
pages=roster51 if roster51==roster53 else sorted(set(roster51)&set(roster53))

## 6. Reconstruct Notebook 51R2 golden masks from its frozen lineage

In [76]:
golden={}
if first_failed_stage is None:
    N45=Path(s51['notebook45']); N50=Path(s51['notebook50'])
    regions=pd.read_parquet(N45/'tables/region_manifest.parquet'); regions['page']=regions.page.astype(str)
    mapinv=pd.read_csv(N50/'tables/m5_map_inventory.csv',dtype={'page':str})
    require({'page','file'}.issubset(mapinv.columns),'Notebook 50 map inventory lacks page/file.')
    def resolve_map(raw):
        p=Path(str(raw)); candidates=[p,N50/p,N50/'maps'/p.name,N50/'masks'/p.name]
        hits={str(x.resolve()):x.resolve() for x in candidates if x.exists()}
        require(len(hits)==1,f'Expected one Notebook 50 map for {raw}; got {list(hits.values())}')
        return list(hits.values())[0]
    def union_m3(page):
        arr=None
        for _,r in regions[regions.page==page].iterrows():
            p=N45/'masks'/page/str(r.region_id).replace(':','_')/'M3.png'; m=read_gray(p)>0
            arr=m.copy() if arr is None else (arr|m)
        require(arr is not None,f'{page}: no Notebook 45 M3 regions')
        return arr
    for page in pages:
        m3=union_m3(page)
        q=mapinv[mapinv.page==page]; require(len(q)==1,f'{page}: Notebook 50 map rows={len(q)}')
        mp=resolve_map(q.iloc[0].file)
        with np.load(mp,allow_pickle=False) as z:
            require('m5_decision_map' in z.files,f'{mp}: no m5_decision_map')
            dm=np.asarray(z['m5_decision_map'])
        require(dm.ndim==2 and dm.shape[0]<=m3.shape[0] and dm.shape[1]<=m3.shape[1],f'{page}: decision canvas exceeds M3')
        padded=np.zeros(m3.shape,dtype=dm.dtype); padded[:dm.shape[0],:dm.shape[1]]=dm

        safe_remove = padded == 1
        safe_protect = padded == 2

        require(
            not np.any(safe_remove & ~m3),
            f'{page}: golden SAFE_REMOVE outside M3'
        )
        require(
            not np.any(safe_protect & ~m3),
            f'{page}: golden SAFE_PROTECT outside M3'
        )
        require(
            not np.any(safe_remove & safe_protect),
            f'{page}: golden decision classes overlap'
        )

        m5_registered_replay = m3 & ~safe_remove & ~safe_protect

        golden[page] = {
            'M3': m3,
            'SAFE_REMOVE': safe_remove,
            'SAFE_PROTECT_DECISION': safe_protect,
            'M5_REGISTERED_REPLAY': m5_registered_replay,
            'map_path': str(mp),
        }

    print('Golden masks reconstructed:',len(golden))


Golden masks reconstructed: 43


## 7. Stages C-E: zero-tolerance M3, SAFE_REMOVE and registered-replay M5 equality

In [77]:
def compare_mask_stage(stage, artifact, filename):
    rows = []

    for page in pages:
        p = N53/'pages'/page/filename
        candidate = read_gray(p) > 0

        golden_array = golden[page][artifact]
        row = exact_row(
            stage,
            page,
            artifact,
            golden_array.astype(np.uint8),
            candidate.astype(np.uint8),
        )
        row['notebook53_path'] = str(p)
        rows.append(row)

    comparison_rows.extend(rows)
    return all(r['exact'] for r in rows), rows

for stage,artifact,key in [
    (stage_order[2], 'M3', 'm3.png'),
    (stage_order[3], 'SAFE_REMOVE', 'safe_remove.png'),
    (stage_order[4], 'M5_REGISTERED_REPLAY', 'm5.png'),
]:
    if can_run(stage):
        passed,rows=compare_mask_stage(stage,artifact,key)
        record_stage(stage,passed,{'pages_compared':len(rows),'different_pixels_total':int(sum(r['different_pixels'] for r in rows)),'failed_pages':[r['page'] for r in rows if not r['exact']]})
    else: record_stage(stage,False,{'not_run_after':first_failed_stage})


## 8. Stage F: exact corrected-page equality against Notebook 51R2 M5 / R2_telea

In [78]:
if can_run(stage_order[5]):
    rows=[]

    gm = inv51[
    inv51.arm.astype(str).str.strip().isin(['M5', 'M5_REMOVE'])
    & (inv51.restorer.astype(str).str.strip() == 'R2_telea')
    ].copy()

    require(
        gm.page.nunique() == len(pages) and len(gm) == len(pages),
        'Notebook 51R2 does not contain exactly one '
        'M5/M5_REMOVE R2_telea candidate per page.'
    )

    for page in pages:
        q=gm[gm.page==page]; require(len(q)==1,f'{page}: golden candidate count {len(q)}')
        gp=Path(str(q.iloc[0].candidate)); require(gp.exists(),f'{page}: missing golden candidate {gp}')
        cp=N53/'pages'/page/'corrected.png'; require(cp.exists(),f'{page}: Notebook 53 corrected page missing')

        eq53 = pd.read_csv(
            N53/'tables/page_equivalence.csv',
            dtype={'page': str}
        )

        require(eq53.N51_exact.astype(bool).all(),
            '53R3 page_equivalence.csv contains a failed page.')

        require(int(eq53.different_pixels.sum()) == 0,
            '53R3 recorded non-zero corrected-page differences.')

        g=read_gray(gp); c=read_gray(cp); row=exact_row(stage_order[5],page,'CORRECTED_M5_R2_TELEA',g,c); row.update({'golden_path':str(gp),'candidate_path':str(cp),'golden_file_sha256':sha256_file(gp),'candidate_file_sha256':sha256_file(cp)})
        rows.append(row)
        if not row['exact'] and g.shape==c.shape:
            diff=np.abs(g.astype(np.int16)-c.astype(np.int16)).astype(np.uint8); cv2.imwrite(str(DIFFS/f'{page}_corrected_absdiff.png'),diff)
    comparison_rows.extend(rows)
    record_stage(stage_order[5],all(r['exact'] for r in rows),{'pages_compared':len(rows),'different_pixels_total':int(sum(r['different_pixels'] for r in rows)),'failed_pages':[r['page'] for r in rows if not r['exact']]})
else: record_stage(stage_order[5],False,{'not_run_after':first_failed_stage})


## 9. Stage G: outside-mask immutability and cross-file internal contract

In [79]:
# ============================================================
# STAGE G
# 51R2 whole-page assembly and registered-replay contracts
# ============================================================

stage = stage_order[6]

# ------------------------------------------------------------
# Resolve the exact Notebook 51R2 source lineage
# ------------------------------------------------------------

source51_path = N51 / 'source_record.json'
require(
    source51_path.exists(),
    f'Missing Notebook 51R2 source record: {source51_path}'
)

source51 = json.loads((source51_path).read_text())

require(
    'notebook45' in source51,
    'Notebook 51R2 source_record.json lacks notebook45.'
)
require(
    'notebook46' in source51,
    'Notebook 51R2 source_record.json lacks notebook46.'
)

N45 = Path(source51['notebook45'])
N46 = Path(source51['notebook46'])

require(
    N45.exists(),
    f'Notebook 45 source run does not exist: {N45}'
)
require(
    N46.exists(),
    f'Notebook 46R6 source run does not exist: {N46}'
)

source45_path = N45 / 'source_record.json'
require(
    source45_path.exists(),
    f'Missing Notebook 45 source record: {source45_path}'
)

source45 = json.loads((source45_path).read_text())

require(
    'source_v82' in source45,
    'Notebook 45 source_record.json lacks source_v82.'
)
require(
    'v75' in source45,
    'Notebook 45 source_record.json lacks v75.'
)

SOURCE_V82 = Path(source45['source_v82'])
V75 = Path(source45['v75'])

require(
    SOURCE_V82.exists(),
    f'Notebook 51R2 observed-page source does not exist: '
    f'{SOURCE_V82}'
)
require(
    V75.exists(),
    f'Notebook 51R2 V7.5 source does not exist: {V75}'
)

# ------------------------------------------------------------
# Resolve the exact authoritative-mask cache selected by 51R2
# ------------------------------------------------------------

cache_status_path = V75 / 'tables' / 'cache_status.csv'
require(
    cache_status_path.exists(),
    f'Missing V7.5 cache status: {cache_status_path}'
)

cache_status = pd.read_csv(
    cache_status_path,
    dtype={'page': str}
)

require(
    {'page', 'cache'}.issubset(cache_status.columns),
    'V7.5 cache_status.csv must contain page and cache columns.'
)

if cache_status.page.duplicated().any():
    duplicates = sorted(
        cache_status.loc[
            cache_status.page.duplicated(keep=False),
            'page'
        ].astype(str).unique()
    )
    require(
        False,
        f'Duplicate V7.5 cache mappings: {duplicates}'
    )

cache_lookup = {
    str(row.page): Path(str(row.cache))
    for _, row in cache_status.iterrows()
}

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def load_51r2_observed(page):
    path = (
        SOURCE_V82
        / 'whole_pages'
        / page
        / 'observed.png'
    )

    observed = read_gray(path)

    require(
        observed is not None,
        f'{page}: missing/unreadable 51R2 observed page: {path}'
    )

    return observed, path


def load_51r2_authority(page):
    require(
        page in cache_lookup,
        f'{page}: no V7.5 cache mapping used by Notebook 51R2.'
    )

    path = cache_lookup[page]

    require(
        path.exists(),
        f'{page}: V7.5 cache used by 51R2 is missing: {path}'
    )

    with np.load(path) as z:
        require(
            'mask' in z.files,
            f'{page}: V7.5 cache lacks mask: {path}'
        )
        authority = z['mask'] > 0

    return authority, path


def load_51r2_protected(page):
    path = (
        N46
        / 'whole_page_candidates'
        / page
        / 'protected_print.png'
    )

    protected_image = read_gray(path)

    require(
        protected_image is not None,
        f'{page}: missing/unreadable Notebook 46R6 '
        f'protected_print.png: {path}'
    )

    return protected_image > 0, path


# ------------------------------------------------------------
# Run exact contracts
# ------------------------------------------------------------

rows = []

for page in pages:
    corrected_path = N53 / 'pages' / page / 'corrected.png'
    reference_path = (
        N53
        / 'pages'
        / page
        / 'reference_51R2.png'
    )

    corrected = read_gray(corrected_path)
    reference_53 = read_gray(reference_path)

    observed, observed_path = load_51r2_observed(page)
    auth, auth_path = load_51r2_authority(page)
    protected, protected_path = load_51r2_protected(page)

    m3 = (
        read_gray(N53 / 'pages' / page / 'm3.png')
        > 0
    )
    safe_remove = (
        read_gray(
            N53 / 'pages' / page / 'safe_remove.png'
        )
        > 0
    )
    safe_protect = (
        read_gray(
            N53
            / 'pages'
            / page
            / 'safe_protect_decision.png'
        )
        > 0
    )
    m5 = (
        read_gray(N53 / 'pages' / page / 'm5.png')
        > 0
    )

    arrays = {
        'corrected': corrected,
        'reference_53': reference_53,
        'observed_51R2': observed,
        'authority': auth,
        'protected': protected,
        'm3': m3,
        'safe_remove': safe_remove,
        'safe_protect': safe_protect,
        'm5': m5,
    }

    shapes = {
        name: value.shape
        for name, value in arrays.items()
    }

    require(
        len(set(shapes.values())) == 1,
        f'{page}: Stage G shape mismatch: {shapes}'
    )

    expected_m5 = (
        m3
        & ~safe_remove
        & ~safe_protect
    )

    outside_authority_drift = int(np.sum(
        corrected[~auth] != observed[~auth]
    ))

    protected_non_residual_drift = int(np.sum(
        corrected[protected & ~m5]
        != observed[protected & ~m5]
    ))

    row = {
        'page': page,

        # Exact Notebook 51R2 whole-page contracts
        'outside_authority_drift':
            outside_authority_drift,

        'protected_non_residual_drift':
            protected_non_residual_drift,

        # Decision-map integrity
        'safe_remove_outside_m3':
            int((safe_remove & ~m3).sum()),

        'safe_protect_outside_m3':
            int((safe_protect & ~m3).sum()),

        'decision_class_overlap':
            int((safe_remove & safe_protect).sum()),

        'm5_registered_replay_formula_xor':
            int(np.logical_xor(
                m5,
                expected_m5
            ).sum()),

        # Final reproduced-page identity
        'corrected_vs_53_reference_xor':
            int(np.sum(
                corrected != reference_53
            )),

        # Source identities for audit
        'observed_path':
            str(observed_path),

        'authority_cache_path':
            str(auth_path),

        'protected_path':
            str(protected_path),

        'corrected_path':
            str(corrected_path),

        'reference_path':
            str(reference_path),
    }

    zero_fields = [
        'outside_authority_drift',
        'protected_non_residual_drift',
        'safe_remove_outside_m3',
        'safe_protect_outside_m3',
        'decision_class_overlap',
        'm5_registered_replay_formula_xor',
        'corrected_vs_53_reference_xor',
    ]

    row['exact'] = all(
        row[field] == 0
        for field in zero_fields
    )

    rows.append(row)

stage_g = pd.DataFrame(rows)

stage_g.to_csv(
    TABLES / 'internal_contracts_by_page.csv',
    index=False
)

stage_g_pass = bool(
    len(stage_g) == len(pages)
    and stage_g.exact.all()
)

comparison_rows.extend([
    {
        'stage': stage,
        'page': row['page'],
        'artifact': '51R2_WHOLE_PAGE_CONTRACTS',
        'exact': bool(row['exact']),
        'different_pixels': int(
            row['outside_authority_drift']
            + row['protected_non_residual_drift']
            + row['safe_remove_outside_m3']
            + row['safe_protect_outside_m3']
            + row['decision_class_overlap']
            + row['m5_registered_replay_formula_xor']
            + row['corrected_vs_53_reference_xor']
        ),
    }
    for row in rows
])

record_stage(
    stage,
    stage_g_pass,
    {
        'pages_tested': int(len(stage_g)),
        'failed_pages': (
            stage_g.loc[
                ~stage_g.exact,
                'page'
            ].astype(str).tolist()
        ),
        'outside_authority_drift': int(
            stage_g.outside_authority_drift.sum()
        ),
        'protected_non_residual_drift': int(
            stage_g.protected_non_residual_drift.sum()
        ),
        'safe_remove_outside_m3': int(
            stage_g.safe_remove_outside_m3.sum()
        ),
        'safe_protect_outside_m3': int(
            stage_g.safe_protect_outside_m3.sum()
        ),
        'decision_class_overlap': int(
            stage_g.decision_class_overlap.sum()
        ),
        'm5_formula_xor': int(
            stage_g
            .m5_registered_replay_formula_xor
            .sum()
        ),
        'corrected_vs_reference_xor': int(
            stage_g.corrected_vs_53_reference_xor.sum()
        ),
        'notebook45_source': str(N45),
        'notebook46_source': str(N46),
        'source_v82': str(SOURCE_V82),
        'v75_source': str(V75),
    }
)

require(
    stage_g_pass,
    f'{stage}: internal 51R2 contract failure. '
    f'See {TABLES / "internal_contracts_by_page.csv"}'
)

## 10. Verdict, first-failure record and final upload package

In [80]:
stages=pd.DataFrame([{'stage':r['stage'],'status':r['status'],'passed':r['passed'],'details_json':json.dumps(r['details'],sort_keys=True,default=str)} for r in stage_records])
stages.to_csv(TABLES/'stage_results.csv',index=False)
comparisons=pd.DataFrame(comparison_rows)
comparisons.to_csv(TABLES/'exact_equivalence_comparisons.csv',index=False)

passed=first_failed_stage is None and len(stage_records)==len(stage_order) and all(r['status']=='PASS' for r in stage_records)
verdict = (
    'PASS_51R2_REGISTERED_REPLAY_EXACT_EQUIVALENCE_'
    'READY_FOR_NOTEBOOK09_SHADOW'
    if passed
    else
    'FAIL_51R2_REGISTERED_REPLAY_EXACT_EQUIVALENCE'
)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.8C-notebook54','verdict':verdict,'first_failed_stage':first_failed_stage,'zero_tolerance':True,'first_failure_semantics':True,'notebook51_run':str(N51),'notebook53_run':str(N53),'pages_in_golden_roster':len(roster51),'pages_in_notebook53_roster':len(roster53),'stages':stage_records,'promotion_authority':False,'meaning':'PASS permits controlled Notebook 09 shadow integration only. FAIL blocks shadow integration and preserves the first failed stage.'}
atomic_json(OUT/'notebook54_verdict.json',record)

environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
identity={'created_utc':record['created_utc'],'version':'8.8C-notebook54-strict-exact-equivalence','notebook51_candidate_lock_id':l51.get('lock_id'),'notebook53_implementation_id':i53.get('implementation_id'),'notebook51_identity_sha256':source_identity['notebook51_identity_sha256'],'notebook53_identity_sha256':source_identity['notebook53_identity_sha256'],'stage_results_sha256':sha256_file(TABLES/'stage_results.csv'),'comparison_table_sha256':sha256_file(TABLES/'exact_equivalence_comparisons.csv'),'zero_tolerance':True,'software':environment}
atomic_json(OUT/'experiment_identity.json',identity); atomic_json(OUT/'software_environment.json',environment)

lines=['NOTEBOOK 54 / V8.8C STRICT EXACT-EQUIVALENCE AUDIT','',f'Verdict: {verdict}',f'First failed stage: {first_failed_stage}',f'Notebook 51R2: {N51}',f'Notebook 53: {N53}',f'Golden pages: {len(roster51)}',f'Notebook 53 pages: {len(roster53)}','',*([f"{r['stage']}: {r['status']}" for r in stage_records]),'','Zero tolerance was used for every mask and page array comparison.','PASS permits controlled shadow integration only; it is not production authority.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))

upload_files=[OUT/'executive_summary.txt',OUT/'notebook54_verdict.json',OUT/'experiment_identity.json',OUT/'source_identity.json',OUT/'software_environment.json',TABLES/'stage_results.csv',TABLES/'exact_equivalence_comparisons.csv']
if (TABLES/'internal_contracts_by_page.csv').exists(): upload_files.append(TABLES/'internal_contracts_by_page.csv')
rows=[]
for src in upload_files:
    require(src.exists(),f'Missing upload artefact {src}')
    dst=UPLOAD/src.name; shutil.copy2(src,dst); rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst),'source':str(src.relative_to(OUT))})
pd.DataFrame(rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 54 V8.8C strict exact-equivalence audit package. Zero-tolerance comparisons and first-failure semantics are authoritative. Difference images remain in the durable run directory.'+chr(10))
zip_path=UPLOAD/'notebook54_v8_8c_exact_equivalence_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p!=zip_path: z.write(p,p.name)
require(zip_path.exists() and zip_path.stat().st_size>0,'Upload bundle creation failed.')
print(chr(10).join(lines)); print('Bundle:',zip_path)
if not passed and RAISE_ON_FAIL: raise RuntimeError(f'Notebook 54 failed closed at {first_failed_stage}. Evidence and upload package were written before stopping.')


NOTEBOOK 54 / V8.8C STRICT EXACT-EQUIVALENCE AUDIT

Verdict: PASS_51R2_REGISTERED_REPLAY_EXACT_EQUIVALENCE_READY_FOR_NOTEBOOK09_SHADOW
First failed stage: None
Notebook 51R2: /content/drive/MyDrive/MusicAnnotationProject/observed_print_classifier_v8_7_notebook51/20261106_080000_v8_7_notebook51R2
Notebook 53: /content/drive/MyDrive/MusicAnnotationProject/observed_print_classifier_v8_8_notebook53R3/20261106_210922_v8_8b_notebook53R3
Golden pages: 43
Notebook 53 pages: 43

A_SOURCE_RUNS_COMPLETE_AND_REGISTERED_REPLAY_ELIGIBLE: PASS
B_PAGE_ROSTER_EXACT: PASS
C_M3_MASK_EXACT: PASS
D_SAFE_REMOVE_DECISION_EXACT: PASS
E_REGISTERED_REPLAY_M5_MASK_EXACT: PASS
F_CORRECTED_PAGE_EXACT_NOTEBOOK51R2_R2_TELEA: PASS
G_REPLAY_IMMUTABILITY_AND_INTERNAL_CONTRACTS: PASS

Zero tolerance was used for every mask and page array comparison.
PASS permits controlled shadow integration only; it is not production authority.
Bundle: /content/drive/MyDrive/MusicAnnotationProject/m5_remove_exact_equivalence_v8_8c/2026

## Outputs to inspect first
1. `executive_summary.txt`
2. `notebook54_verdict.json`
3. `tables/stage_results.csv`
4. `tables/exact_equivalence_comparisons.csv`
5. `tables/internal_contracts_by_page.csv`, if Stage G ran
6. `difference_images/`, if corrected-page equality failed
7. `upload/notebook54_v8_8c_exact_equivalence_bundle.zip`

A failure is not softened by later stages. The first failed stage is the audit decision boundary.
